# Laboratorium 1: Perceptron wielowarstwowy i pętla treningowa

Dane: Fashion-MNIST (`keras.datasets.fashion_mnist`).

Zaczynamy od regresji softmax jako punktu odniesienia, potem budujemy perceptron wielowarstwowy (MLP) w Keras, trenujemy go przez `fit` i drugi raz własną pętlą z `tf.GradientTape`. Dalej sprawdzamy wpływ learning rate, rozmiaru batcha i funkcji aktywacji, a na końcu raz oceniamy model na zbiorze testowym i oglądamy jego błędy. Sekcja "Zadania rozszerzone" na końcu notebooka dotyczy inicjalizacji wag, BatchNormalization i zanikającego gradientu.

Środowisko: Google Colab z GPU (Środowisko wykonawcze, Zmień typ środowiska wykonawczego, T4 GPU). Keras 3 na backendzie TensorFlow. Wszystkie potrzebne biblioteki są w Colabie zainstalowane.

In [ ]:
import time

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
keras.utils.set_random_seed(RANDOM_STATE)

print("Keras", keras.__version__, "| backend", keras.backend.backend(), "| TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "brak, obliczenia na CPU")

In [ ]:
# Parametry przebiegu (te same w Colabie i w rozwiązaniu wzorcowym)
EPOKI_BASELINE = 10   # regresja softmax, zadanie 2
EPOKI_MLP = 15        # MLP trenowany przez fit, zadanie 3
EPOKI_PETLA = 3       # własna pętla treningowa, zadanie 4
EPOKI_EKSP = 5        # każdy przebieg w eksperymentach (zadania 5, 8, 9, 10)
BATCH = 128
UDZIAL_WALIDACJI = 0.1

## Zadanie 1: Dane

Fashion-MNIST to 70 000 obrazów 28x28 w skali szarości z dziesięcioma kategoriami odzieży i obuwia (60 000 treningowych i 10 000 testowych). Format jest taki sam jak w MNIST, ale zadanie jest trudniejsze, bo kilka klas wygląda podobnie. Keras pobiera zbiór przy pierwszym wywołaniu `load_data()`.

Kroki: kształty i zakres wartości, rozkład klas, siatka przykładów z nazwami klas, normalizacja do [0, 1] i wydzielenie 10% zbioru treningowego na walidację.

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Nazwy klas w kolejności etykiet 0 do 9 (w nawiasach nazwy z dokumentacji Keras):
# T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot
KLASY = ["koszulka", "spodnie", "sweter", "sukienka", "płaszcz",
         "sandał", "koszula", "but sportowy", "torba", "botek"]

In [ ]:
def pokaz_obrazy(obrazy, podpisy, n_kolumn=10, rozmiar=1.4):
    """Rysuje siatkę obrazów 28x28 z podpisami."""
    n = len(obrazy)
    n_wierszy = int(np.ceil(n / n_kolumn))
    fig, osie = plt.subplots(n_wierszy, n_kolumn, squeeze=False,
                             figsize=(rozmiar * n_kolumn, rozmiar * 1.3 * n_wierszy))
    for i, ax in enumerate(osie.flat):
        ax.axis("off")
        if i < n:
            ax.imshow(obrazy[i], cmap="gray_r")
            ax.set_title(podpisy[i], fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# TODO 1.1: wypiszcie kształty i typy tablic x_train_full, y_train_full, x_test, y_test
# oraz najmniejszą i największą wartość piksela.

In [ ]:
# TODO 1.2: policzcie przykłady każdej klasy w zbiorze treningowym i testowym (np. np.bincount)
# i pokażcie je w tabeli pandas z nazwami z listy KLASY.

In [ ]:
# TODO 1.3: narysujcie po dwa losowe przykłady z każdej klasy funkcją pokaz_obrazy,
# z podpisami z listy KLASY.

In [ ]:
# TODO 1.4: zamieńcie piksele na float32 w zakresie [0, 1] i wydzielcie z treningu walidację:
# train_test_split(..., test_size=UDZIAL_WALIDACJI, stratify=y_train_full, random_state=RANDOM_STATE).
# Zbiór testowy też znormalizujcie (x_test_n), ale nie używajcie go aż do zadania 6.
x_train, x_val, y_train, y_val = None, None, None, None  # TODO
x_test_n = None  # TODO

### Pytania do przemyślenia

1. Dlaczego dzielimy piksele przez 255? Co by się zmieniło, gdyby sieć dostawała wartości od 0 do 255?
2. Po co wydzielać walidację z treningu, skoro zbiór testowy jest gotowy?
3. Czy klasy są zrównoważone i co z tego wynika dla dokładności (accuracy) jako miary jakości?

Odpowiedzi:

1. 
2. 
3. 

## Zadanie 2: Baseline, regresja softmax

Najprostszy sensowny model: `Flatten` zamienia obraz na wektor 784 liczb, a jedna warstwa `Dense(10, activation="softmax")` zwraca prawdopodobieństwa dziesięciu klas. To wieloklasowa regresja logistyczna zapisana jako sieć bez warstw ukrytych. Jej wynik będzie punktem odniesienia dla MLP.

Kroki: model `keras.Sequential`, kompilacja (Adam z learning rate 1e-3, strata `sparse_categorical_crossentropy`, metryka `accuracy`), ocena przed treningiem, trening przez `EPOKI_BASELINE` epok z `validation_data`, krzywe uczenia i dokładność walidacyjna.

In [ ]:
def rysuj_krzywe(historia, tytul):
    """Krzywe uczenia z obiektu History: strata i dokładność na treningu i walidacji."""
    h = historia.history
    epoki = np.arange(1, len(h["loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
    ax1.plot(epoki, h["loss"], "o-", label="trening")
    ax1.plot(epoki, h["val_loss"], "o-", label="walidacja")
    ax1.set(xlabel="epoka", ylabel="strata (entropia krzyżowa)", title=f"{tytul}: strata")
    ax2.plot(epoki, h["accuracy"], "o-", label="trening")
    ax2.plot(epoki, h["val_accuracy"], "o-", label="walidacja")
    ax2.set(xlabel="epoka", ylabel="dokładność", title=f"{tytul}: dokładność")
    for ax in (ax1, ax2):
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# TODO 2.1: keras.utils.set_random_seed(RANDOM_STATE), potem model_baseline: Input((28, 28)), Flatten, Dense(10, "softmax").
# Kompilacja: optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="sparse_categorical_crossentropy",
# metrics=["accuracy"] (podawajcie argumenty z nazwami). Wypiszcie model_baseline.summary().
model_baseline = None  # TODO

In [ ]:
# TODO 2.2: przed treningiem policzcie model_baseline.evaluate(x_val, y_val, verbose=0)
# i porównajcie stratę z np.log(10).

In [ ]:
# TODO 2.3: trening: fit(x_train, y_train, validation_data=(x_val, y_val), epochs=EPOKI_BASELINE,
# batch_size=BATCH, verbose=2). Zmierzcie czas (time.perf_counter()).
historia_baseline = None  # TODO

In [ ]:
# TODO 2.4: krzywe uczenia (rysuj_krzywe) i dokładność walidacyjna w ostatniej epoce (acc_baseline)
acc_baseline = None  # TODO

### Pytania do przemyślenia

1. Ile parametrów ma model i skąd się bierze ta liczba?
2. Jaką stratę miał model przed treningiem i dlaczego jest ona bliska ln 10?
3. Czy krzywe treningowa i walidacyjna się rozjeżdżają? Co to mówi o pojemności tego modelu?

Odpowiedzi:

1. 
2. 
3. 

## Zadanie 3: Perceptron wielowarstwowy

Kroki: funkcja `zbuduj_mlp(aktywacja="relu", nazwa="mlp")` zwracająca nieskompilowany model (Flatten, Dense(256), Dense(128), Dense(10, softmax)), która przyda się w zadaniach 4 i 5. Kompilacja jak w zadaniu 2, trening przez `EPOKI_MLP` epok, `model.summary()` i ręczne policzenie parametrów pierwszej warstwy ukrytej, krzywe uczenia i epoka z najlepszym wynikiem na walidacji.

In [ ]:
# TODO 3.1: funkcja zbuduj_mlp zwracająca nieskompilowany keras.Sequential:
# Input((28, 28)), Flatten, Dense(256, activation=aktywacja), Dense(128, activation=aktywacja), Dense(10, "softmax")
def zbuduj_mlp(aktywacja="relu", nazwa="mlp"):
    return None  # TODO

In [ ]:
# TODO 3.2: ziarno, model_mlp = zbuduj_mlp(), kompilacja jak w zadaniu 2, model_mlp.summary(),
# trening przez EPOKI_MLP epok (verbose=2) z pomiarem czasu.
model_mlp = None  # TODO
historia_mlp = None  # TODO

In [ ]:
# TODO 3.3: policzcie ręcznie parametry pierwszej warstwy Dense (wagi i biasy) oraz całego modelu
# i porównajcie z count_params(). Uwaga: model_mlp.layers nie zawiera warstwy Input.
# Ile razy więcej parametrów ma MLP niż baseline?

In [ ]:
# TODO 3.4: krzywe uczenia (rysuj_krzywe); epoka z najlepszą val_accuracy i epoka z najmniejszą val_loss;
# strata i dokładność treningowa w ostatniej epoce.

### Pytania do przemyślenia

1. Ile parametrów ma MLP i jaka ich część przypada na pierwszą warstwę ukrytą? Ile dokładności zyskaliście względem baseline?
2. Od której epoki krzywe straty treningowej i walidacyjnej się rozjeżdżają i co to oznacza? Jak to możliwe, że dokładność walidacyjna jeszcze nie spada, choć strata walidacyjna już rośnie?
3. Którą epokę wybralibyście na model końcowy i jak zautomatyzować ten wybór?
4. Co by się stało, gdyby warstwy ukryte nie miały funkcji aktywacji?

Odpowiedzi:

1. 
2. 
3. 
4. 

## Zadanie 4: Własna pętla treningowa z `tf.GradientTape`

`fit` w każdym kroku robi to samo: przepuszcza batch przez model, liczy stratę, wyznacza gradienty straty po wagach, aktualizuje wagi optymalizatorem i odświeża metryki. Tutaj zapiszecie to jawnie.

Ten fragment jest specyficzny dla backendu TensorFlow. Gradienty liczy `tf.GradientTape`, dane podaje `tf.data`, a `tf.function` kompiluje krok do grafu obliczeń. Na backendzie JAX gradienty liczy się funkcją `jax.value_and_grad`, a w PyTorch przez `loss.backward()`, więc ten kod nie przeniesie się bez zmian. Model, optymalizator, funkcja straty i metryki pochodzą z Keras i są wspólne dla wszystkich backendów.

Kroki: zbiory `tf.data`, model z `zbuduj_mlp`, optymalizator, funkcja straty i metryki, funkcje `krok_treningowy` i `krok_walidacyjny`, pętla przez `EPOKI_PETLA` epok, porównanie z `fit` przy tych samych ustawieniach, pomiar czasu kroku z `tf.function` i bez.

In [ ]:
# TODO 4.1: ds_train = tf.data.Dataset.from_tensor_slices((x_train, y_train)), potem
# .shuffle(len(x_train), seed=RANDOM_STATE).batch(BATCH); ds_val podobnie, bez tasowania, batch 1024.
ds_train = None  # TODO
ds_val = None  # TODO

In [ ]:
# TODO 4.2: ziarno; model_petla = zbuduj_mlp(nazwa="mlp_petla"); optymalizator = keras.optimizers.Adam(1e-3);
# funkcja_straty = keras.losses.SparseCategoricalCrossentropy();
# metryki: srednia_straty = keras.metrics.Mean(), acc_train i acc_val = keras.metrics.SparseCategoricalAccuracy()

In [ ]:
# TODO 4.3: krok_treningowy(x, y): w bloku `with tf.GradientTape() as tasma:` predykcja z training=True i strata;
# potem gradienty = tasma.gradient(strata, model_petla.trainable_weights),
# optymalizator.apply_gradients(zip(gradienty, model_petla.trainable_weights)) i aktualizacja
# srednia_straty oraz acc_train. krok_walidacyjny(x, y): predykcja z training=False i aktualizacja acc_val.
def krok_treningowy(x, y):
    pass  # TODO

def krok_walidacyjny(x, y):
    pass  # TODO

# Wersje skompilowane do grafu TensorFlow
krok_treningowy_tf = tf.function(krok_treningowy)
krok_walidacyjny_tf = tf.function(krok_walidacyjny)

In [ ]:
# TODO 4.4: pętla po EPOKI_PETLA epokach: na początku epoki reset_state() wszystkich metryk,
# krok_treningowy_tf dla każdego batcha z ds_train, krok_walidacyjny_tf dla ds_val;
# do listy wyniki_petla dopiszcie słownik z epoką, stratą, dokładnością, val_accuracy i czasem epoki.
wyniki_petla = []  # TODO

In [ ]:
# TODO 4.5: ten sam model trenowany przez fit: ziarno, zbuduj_mlp(nazwa="mlp_fit"), kompilacja jak w zadaniu 3,
# EPOKI_PETLA epok. Tabela: epoka, strata i val_accuracy z pętli i z fit, oraz łączny czas obu sposobów.

In [ ]:
# TODO 4.6: partie = list(ds_train.take(100)); zmierzcie czas 100 wywołań krok_treningowy_tf
# i 100 wywołań krok_treningowy (tryb eager, bez tf.function) na tych samych batchach.
# Uwaga: oba pomiary dalej trenują model_petla, to tutaj nie przeszkadza.

### Pytania do przemyślenia

1. Które linie waszej pętli odpowiadają za forward pass, za backpropagation, a które za aktualizację wag? Co z tego robi `tf.GradientTape`, a co optymalizator?
2. Dlaczego wyniki pętli i `fit` nie są identyczne, choć architektura, ziarno i hiperparametry są te same?
3. Po co argument `training=True` w kroku treningowym, skoro ten MLP działa tak samo w obu trybach?
4. Skąd różnica czasu między `tf.function` a trybem eager? Kiedy własna pętla jest potrzebna, a kiedy wystarczy `fit`?

Odpowiedzi:

1. 
2. 
3. 
4. 

## Zadanie 5: Eksperymenty z hiperparametrami

Każdy przebieg trwa `EPOKI_EKSP` epok i startuje z tego samego ziarna, więc różnice wynikają z jednego zmienianego parametru (i z niedeterminizmu obliczeń). Konfiguracja bazowa: Adam, learning rate 1e-3, batch 128, ReLU. Policzcie ją raz i używajcie we wszystkich trzech porównaniach.

Kroki: funkcja `eksperyment(lr, batch, aktywacja)`, przebiegi dla learning rate 1e-4, 1e-3, 1e-2 i 1e-1, batcha 32, 128 i 256 oraz aktywacji relu, sigmoid i tanh, tabela wyników i wykres dokładności walidacyjnej.

In [ ]:
def rysuj_porownanie(ax, historie, tytul):
    """historie: słownik {etykieta: History}; rysuje dokładność walidacyjną po kolejnych epokach."""
    for etykieta, historia in historie.items():
        acc = historia.history["val_accuracy"]
        ax.plot(np.arange(1, len(acc) + 1), acc, "o-", label=etykieta)
    ax.set(xlabel="epoka", ylabel="dokładność walidacyjna", title=tytul)
    ax.grid(alpha=0.3)
    ax.legend()

In [ ]:
# TODO 5.1: eksperyment(lr=1e-3, batch=BATCH, aktywacja="relu"): ziarno, zbuduj_mlp(aktywacja),
# kompilacja z Adam(learning_rate=lr), fit przez EPOKI_EKSP epok z verbose=0.
# Zwraca (historia, czas treningu w sekundach).
def eksperyment(lr=1e-3, batch=BATCH, aktywacja="relu"):
    return None, 0.0  # TODO

In [ ]:
# TODO 5.2: przebiegi w trzech grupach (learning rate, batch, aktywacja); konfigurację bazową liczcie raz.
# historie_eksp = {"learning rate": {"1e-4": History, ...}, "batch": {...}, "aktywacja": {...}}.
# Tabela wyniki_eksp: grupa, wartość, kroki na epokę, val_acc w ostatniej epoce, najlepsza val_acc,
# strata treningowa i walidacyjna w ostatniej epoce, czas.
historie_eksp = {}  # TODO
wyniki_eksp = pd.DataFrame()  # TODO

In [ ]:
# TODO 5.3: wykres z trzema panelami (fig, osie = plt.subplots(1, 3, figsize=(16, 4))),
# w każdym rysuj_porownanie dla jednej grupy z historie_eksp.

### Pytania do przemyślenia

1. Co się dzieje przy learning rate 1e-1, a co przy 1e-4? Czego spodziewacie się po tych dwóch przebiegach po 50 epokach?
2. Dlaczego sigmoid w warstwach ukrytych uczy się wolniej niż ReLU i tanh?
3. Batch 32 i 256: ile aktualizacji wag wykonuje każdy wariant w tych samych epokach, ile trwa trening i czy porównanie przy tej samej liczbie epok jest uczciwe?
4. Czy różnicę 0,3 punktu procentowego między dwoma przebiegami uznalibyście za istotną? Co trzeba by zrobić, żeby to sprawdzić?

Odpowiedzi:

1. 
2. 
3. 
4. 

## Zadanie 6: Ocena na zbiorze testowym i analiza błędów

Modelem końcowym jest MLP z zadania 3. Wybraliście go na podstawie walidacji, więc dopiero teraz, jeden raz, liczycie dokładność na zbiorze testowym. Potem sprawdzacie, gdzie i jak model się myli.

Kroki: dokładność i strata na teście, macierz pomyłek, trafność w każdej klasie i najczęściej mylone pary, 12 błędnych predykcji, przy których model był najbardziej pewny.

In [ ]:
def rysuj_macierz_pomylek(cm, klasy):
    """Macierz pomyłek z liczbami w komórkach (wiersze: klasa prawdziwa, kolumny: przewidziana)."""
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(klasy)), klasy, rotation=45, ha="right")
    ax.set_yticks(range(len(klasy)), klasy)
    prog = cm.max() / 2
    for i in range(len(klasy)):
        for j in range(len(klasy)):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                    color="white" if cm[i, j] > prog else "black")
    ax.set(xlabel="klasa przewidziana", ylabel="klasa prawdziwa", title="Macierz pomyłek (zbiór testowy)")
    fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()

In [ ]:
# TODO 6.1: model_mlp.evaluate(x_test_n, y_test, verbose=0): strata i dokładność na teście;
# porównajcie z val_accuracy z ostatniej epoki zadania 3.

In [ ]:
# TODO 6.2: proby = model_mlp.predict(x_test_n, verbose=0); y_pred = proby.argmax(axis=1);
# cm = confusion_matrix(y_test, y_pred); rysuj_macierz_pomylek(cm, KLASY)

In [ ]:
# TODO 6.3: trafność w każdej klasie (przekątna cm podzielona przez sumę wiersza), posortowana rosnąco;
# sześć najczęstszych par (klasa prawdziwa, klasa przewidziana) poza przekątną.

In [ ]:
# TODO 6.4: wśród błędnych predykcji znajdźcie 12 z największym prawdopodobieństwem przypisanym
# (błędnej) klasie i narysujcie je funkcją pokaz_obrazy (n_kolumn=6, rozmiar=1.9) z podpisem
# zawierającym klasę prawdziwą, przewidzianą i prawdopodobieństwo. Ile błędów ma pewność powyżej 0,99?

### Pytania do przemyślenia

1. Jak dokładność testowa ma się do walidacyjnej? Dlaczego zbiór testowy wykorzystujemy raz, na końcu?
2. Które klasy model myli najczęściej i dlaczego akurat te?
3. Obejrzyjcie najbardziej pewne błędy. Czy któreś z nich wyglądają raczej na wątpliwą etykietę niż na błąd modelu? Co oznacza pewność 0,99 przy błędnej odpowiedzi?
4. Co w budowie MLP utrudnia odróżnienie koszuli od koszulki i swetra? Co zmieni sieć konwolucyjna z laboratorium 2?

Odpowiedzi:

1. 
2. 
3. 
4. 

## Zadanie 7: Pytania kontrolne

1. Czym różni się regresja softmax od MLP i dlaczego MLP bez nieliniowych funkcji aktywacji nie jest od niej silniejszy?
2. Ile parametrów ma warstwa `Dense` z n wejściami i m neuronami?
3. Wymieńcie kroki jednej iteracji pętli treningowej. Które z nich wykonuje `tf.GradientTape`, a które optymalizator?
4. Po czym na krzywych uczenia poznacie przeuczenie, a po czym zbyt duży learning rate?
5. Dlaczego hiperparametry dobieramy na walidacji, a zbioru testowego używamy raz, na końcu?
6. Dlaczego sigmoid w warstwach ukrytych uczy się wolniej niż ReLU?
7. Czym różni się epoka od kroku (batcha)? Ile kroków ma epoka przy 54 000 przykładach i batchu 128?

Odpowiedzi:

1. 
2. 
3. 
4. 
5. 
6. 
7. 

# Zadania rozszerzone

Zadania 8 do 11 wykraczają poza podstawowy zakres laboratorium. Korzystają z danych i funkcji z zadań 1 do 5.

## Zadanie 8: Inicjalizacja wag

Domyślnie warstwa `Dense` losuje wagi z rozkładu Glorota (`glorot_uniform`), dobranego tak, żeby wariancja sygnału nie rosła ani nie malała z warstwy na warstwę. Dla ReLU proponowana jest inicjalizacja He (`he_normal`). Trzeci wariant to rozkład normalny o odchyleniu standardowym 0,001, czyli wagi zdecydowanie za małe.

Kroki: funkcja `zbuduj_mlp_rozsz(inicjalizacja, bn, aktywacja)` (parametr `bn` przyda się w zadaniu 9), odchylenie standardowe wyjść warstw przed treningiem, trening z SGD (learning rate 0,01, bez momentum) przez `EPOKI_EKSP` epok dla trzech inicjalizacji, a dla zbyt małych wag dodatkowo z optymalizatorem Adam.

In [ ]:
INICJALIZACJE = ["glorot_uniform", "he_normal", "normal_0.001"]

def nowa_inicjalizacja(nazwa):
    """Zwraca nowy obiekt inicjalizatora. Każda warstwa powinna dostać własny obiekt: jeden obiekt
    użyty w kilku warstwach o tym samym kształcie daje im identyczne wagi początkowe."""
    if nazwa == "normal_0.001":
        return keras.initializers.RandomNormal(stddev=0.001)
    return keras.initializers.get(nazwa)

def odchylenia_aktywacji(model, x):
    """Odchylenie standardowe wyjścia każdej warstwy Dense przy jednym przejściu w przód (bez treningu)."""
    h = keras.ops.convert_to_tensor(x)
    wynik = []
    for warstwa in model.layers:
        h = warstwa(h, training=False)
        if isinstance(warstwa, keras.layers.Dense):
            wynik.append(float(keras.ops.std(h)))
    return wynik

In [ ]:
# TODO 8.1: zbuduj_mlp_rozsz(inicjalizacja="glorot_uniform", bn=False, aktywacja="relu", nazwa="mlp_rozsz").
# Bez BN: jak zbuduj_mlp, ale każda warstwa Dense dostaje kernel_initializer=nowa_inicjalizacja(inicjalizacja).
# Z BN (zadanie 9): dla warstw ukrytych Dense(n, use_bias=False, kernel_initializer=...), BatchNormalization(),
# Activation(aktywacja). Warstwa wyjściowa w obu wariantach: Dense(10, "softmax") z tą samą inicjalizacją.
def zbuduj_mlp_rozsz(inicjalizacja="glorot_uniform", bn=False, aktywacja="relu", nazwa="mlp_rozsz"):
    return None  # TODO

In [ ]:
# TODO 8.2: dla każdej inicjalizacji z INICJALIZACJE (ziarno przed budową modelu) policzcie
# odchylenia_aktywacji(model, x_val[:256]) i pokażcie je w tabeli (wiersze: ukryta 1, ukryta 2, wyjście).

In [ ]:
# TODO 8.3: trenuj_rozsz(model, optymalizator, epoki=EPOKI_EKSP): kompilacja, strata na walidacji przed treningiem
# (evaluate), fit z verbose=0; zwraca (historia, strata_przed, czas).
# Przebiegi: trzy inicjalizacje z keras.optimizers.SGD(learning_rate=0.01) oraz "normal_0.001" z Adam(1e-3).
# Tabela: inicjalizacja, optymalizator, strata przed treningiem, strata po 1. epoce, val_acc w ostatniej epoce.
# Wykres: rysuj_porownanie dla czterech przebiegów.

### Pytania do przemyślenia

1. Jak odchylenie standardowe wyjść warstw zależy od inicjalizacji? Co dzieje się z sygnałem przy wagach o odchyleniu 0,001?
2. Dlaczego SGD przy zbyt małych wagach praktycznie stoi w miejscu, a Adam przy tych samych wagach się uczy?
3. Czy różnica między inicjalizacją Glorota i He ma znaczenie w tej sieci? Kiedy mogłaby mieć?

Odpowiedzi:

1. 
2. 
3. 

## Zadanie 9: BatchNormalization

`BatchNormalization` sprowadza wyjście warstwy do średniej 0 i wariancji 1 w obrębie batcha, a potem skaluje je i przesuwa wyuczonymi parametrami gamma i beta. W czasie treningu korzysta ze statystyk bieżącego batcha, a w czasie predykcji ze średnich kroczących zebranych podczas treningu. Dlatego tutaj argument `training` z zadania 4 ma znaczenie. Warstwa Dense przed BatchNormalization nie potrzebuje biasu (`use_bias=False`), bo przesunięcie i tak daje beta.

Kroki: liczba parametrów modelu z BN (trenowalne i nietrenowalne), trening z BN przy zbyt małych wagach i SGD (porównanie z zadaniem 8), trening z BN przy domyślnej inicjalizacji i optymalizatorze Adam (porównanie z konfiguracją bazową z zadania 5).

In [ ]:
# TODO 9.1: model z BN (zbuduj_mlp_rozsz(bn=True)), summary(), liczba wag trenowalnych i nietrenowalnych
# (model.trainable_weights, model.non_trainable_weights, np.prod(w.shape)).

In [ ]:
# TODO 9.2: dwa przebiegi z BN: "normal_0.001" z SGD(0.01) oraz "glorot_uniform" z Adam(1e-3).
# Tabela razem z odpowiednikami bez BN (z zadania 8 i konfiguracją bazową z zadania 5) i wykres.

### Pytania do przemyślenia

1. Ile parametrów dodaje BatchNormalization i które z nich nie są trenowane przez optymalizator?
2. Dlaczego BatchNormalization ratuje trening przy zbyt małych wagach i SGD?
3. Czy BatchNormalization poprawił wynik konfiguracji bazowej w tej płytkiej sieci? Jak oceniacie wiarygodność tej różnicy?

Odpowiedzi:

1. 
2. 
3. 

## Zadanie 10: Zanikający gradient

W sieci z wieloma warstwami gradient straty po wagach pierwszych warstw jest iloczynem wielu czynników, po jednym na każdą warstwę nad nimi. Pochodna sigmoidy nie przekracza 0,25, więc przy tej aktywacji iloczyn szybko maleje. Sprawdzicie to wprost: zmierzycie normę gradientu jądra (`kernel`) każdej warstwy Dense na jednym batchu, przed treningiem.

Kroki: funkcja `normy_gradientow(model, x, y)` z `tf.GradientTape` (fragment dla backendu TensorFlow), normy dla sieci z 10 warstwami ukrytymi Dense(64) z aktywacją sigmoid, tanh i relu oraz z sigmoidą i BatchNormalization, wykres w skali logarytmicznej, krótki trening (`EPOKI_EKSP` epok, Adam) dla sigmoid, relu i sigmoid z BN.

In [ ]:
def zbuduj_gleboka(aktywacja, liczba_warstw=10, szerokosc=64, bn=False):
    """Głęboki MLP: liczba_warstw warstw ukrytych Dense(szerokosc), opcjonalnie z BatchNormalization."""
    warstwy = [keras.Input(shape=(28, 28)), keras.layers.Flatten()]
    for _ in range(liczba_warstw):
        if bn:
            warstwy += [keras.layers.Dense(szerokosc, use_bias=False),
                        keras.layers.BatchNormalization(),
                        keras.layers.Activation(aktywacja)]
        else:
            warstwy.append(keras.layers.Dense(szerokosc, activation=aktywacja))
    warstwy.append(keras.layers.Dense(10, activation="softmax"))
    return keras.Sequential(warstwy, name=f"gleboka_{aktywacja}" + ("_bn" if bn else ""))

WARIANTY_GLEBOKIE = {"sigmoid": dict(aktywacja="sigmoid"), "tanh": dict(aktywacja="tanh"),
                     "relu": dict(aktywacja="relu"), "sigmoid + BN": dict(aktywacja="sigmoid", bn=True)}

In [ ]:
# TODO 10.1: normy_gradientow(model, x, y): lista jąder warstw Dense ([w.kernel for w in model.layers if
# isinstance(w, keras.layers.Dense)]), w bloku tf.GradientTape strata
# keras.ops.mean(keras.losses.sparse_categorical_crossentropy(y, model(x, training=True))),
# gradienty po jądrach i ich normy (float(tf.norm(g))).
def normy_gradientow(model, x, y):
    return []  # TODO

In [ ]:
# TODO 10.2: dla każdego wariantu z WARIANTY_GLEBOKIE (ziarno przed budową) policzcie normy na batchu
# x_train[:256], y_train[:256]. Tabela (wiersze: warstwy 1 do 11) i wykres: ax.semilogy(numer warstwy, normy).
# Dla każdego wariantu podajcie stosunek normy w warstwie wyjściowej do normy w pierwszej warstwie.
normy_glebokie = {}  # TODO

In [ ]:
# TODO 10.3: trening (trenuj_rozsz, Adam 1e-3, EPOKI_EKSP epok) dla wariantów sigmoid, relu i sigmoid + BN;
# tabela z val_acc i wykres rysuj_porownanie.

### Pytania do przemyślenia

1. Ile razy norma gradientu w warstwie wyjściowej jest większa niż w pierwszej warstwie przy sigmoidzie, a ile przy ReLU?
2. Jak ten pomiar przekłada się na trening głębokiej sieci z sigmoidą? Dlaczego nie pomaga nawet Adam?
3. Dlaczego tanh, choć też się nasyca, wypada tu znacznie lepiej niż sigmoid? Co zmienia BatchNormalization?

Odpowiedzi:

1. 
2. 
3. 

## Zadanie 11: Wnioski

Który z trzech mechanizmów z zadań 8 do 10 (inicjalizacja wag, BatchNormalization, wybór funkcji aktywacji) zastosowalibyście najpierw w nowej, głębokiej sieci i dlaczego? Odpowiedzcie w dwóch, trzech zdaniach, odwołując się do własnych wyników.

Odpowiedź: